In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Projects that used EvoSuite test generation

In [ ]:
import pandas as pd

query = f"SELECT root_path, runtime FROM project AS p WHERE p.use_test_generation = true"
df = pd.read_sql_query(query, conn)
df

## Runtime requirements for EvoSuite test generation per project

In [ ]:
import pandas as pd

query = f"""
SELECT 
    rt.project_id,
    project_name(rt.project_id) AS project_name,
    sum(rt.runtime) AS runtime
FROM 
    project AS p
    INNER JOIN evosuite_runtime AS rt ON p.id = rt.project_id
WHERE 
    p.use_test_generation = true
GROUP BY 
    rt.project_id
"""
df = pd.read_sql_query(query, conn)
df


## Runtime requirements for EvoSuite test generation per class / phase

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# New query to get class-level phase runtime data
query = """
SELECT 
    (p.configuration::json->'evosuite'->>'search_budget')::int AS search_budget,
    e.class_name,
    e.phase_name,
    e.step,
    e.runtime
FROM 
    project AS p
    INNER JOIN evosuite_runtime AS e ON p.id = e.project_id
WHERE 
    p.use_test_generation = true
    AND e.class_name IS NOT NULL
"""

# Execute the query
df = pd.read_sql_query(query, conn)

# Define the correct phase order
correct_phase_order = [
    'SEARCH',
    'INLINING',
    'MINIMIZATION',
    'COVERAGE_ANALYSIS',
    'ASSERTION_GENERATION',
    'JUNIT_CHECK',
    'WRITING_TESTS',
    'WRITING_STATISTICS',
    'DONE',
    'FINISHED'
]

# Calculate MEAN runtime per phase per class per search budget
mean_phase_runtimes = df.groupby(['search_budget', 'phase_name'])['runtime'].mean().reset_index()

# Create a pivot table to show mean runtime per phase for each search budget
result_mean = mean_phase_runtimes.pivot_table(
    index='search_budget',
    columns='phase_name',
    values='runtime',
    aggfunc='mean'
).fillna(0)

# Calculate MEDIAN runtime per phase per class per search budget
median_phase_runtimes = df.groupby(['search_budget', 'phase_name'])['runtime'].median().reset_index()

# Create a pivot table to show median runtime per phase for each search budget
result_median = median_phase_runtimes.pivot_table(
    index='search_budget',
    columns='phase_name',
    values='runtime',
    aggfunc='mean'  # This is just to create the pivot, the values are already medians
).fillna(0)

# Get all available phases in the data
available_phases = set(result_mean.columns)

# Create a list of phases in the correct order, including only those that exist in the data
ordered_phases = [phase for phase in correct_phase_order if phase in available_phases]

# Add any phases that might be in the data but not in our predefined list (at the end)
extra_phases = [phase for phase in available_phases if phase not in correct_phase_order]
ordered_phases.extend(sorted(extra_phases))

# Reorder the columns based on the correct phase order for both dataframes
result_mean = result_mean[ordered_phases]
result_median = result_median[ordered_phases]

# Calculate total runtime per search budget for both mean and median
total_mean = result_mean.sum(axis=1)
total_median = result_median.sum(axis=1)

# Create new dataframes with total first, then all phases
result_mean_with_total = pd.DataFrame()
result_mean_with_total['<TOTAL>'] = total_mean
for phase in ordered_phases:
    result_mean_with_total[phase] = result_mean[phase]

result_median_with_total = pd.DataFrame()
result_median_with_total['<TOTAL>'] = total_median
for phase in ordered_phases:
    result_median_with_total[phase] = result_median[phase]

# Format the runtimes to be more readable (in seconds with 2 decimal places)
result_mean_formatted = result_mean_with_total.map(lambda x: f"{x:.2f}s")
result_median_formatted = result_median_with_total.map(lambda x: f"{x:.2f}s")

# Display the results with total first
print("MEAN runtime (in seconds) per phase per class, by search budget:")
display(result_mean_formatted)

print("\nMEDIAN runtime (in seconds) per phase per class, by search budget:")
display(result_median_formatted)

# Create bar chart only for the mean runtimes
plt.figure(figsize=(16, 10))
all_columns = ['<TOTAL>'] + ordered_phases
x = np.arange(len(all_columns))
width = 0.8 / len(result_mean_with_total.index)
search_budgets = result_mean_with_total.index.tolist()

for i, budget in enumerate(search_budgets):
    offset = (i - len(search_budgets)/2 + 0.5) * width
    plt.bar(x + offset, result_mean_with_total.loc[budget], width, label=f'Budget: {budget}')

plt.xlabel('Phase')
plt.ylabel('Average Runtime (seconds)')
plt.title('MEAN Runtime per Phase per Class by Search Budget')
plt.xticks(x, all_columns, rotation=45, ha='right')
plt.legend(title='Search Budget')
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

## Fraction of runtime spent in different EvoSuite phases

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Query to get the necessary data including project_id and step
query = """
SELECT (p.configuration::json->'evosuite'->>'search_budget')::int AS search_budget, 
       e.project_id,
       e.phase_name, 
       e.step,
       e.runtime 
FROM project AS p
INNER JOIN evosuite_runtime AS e ON p.id = e.project_id
WHERE p.use_test_generation = true
"""
df = pd.read_sql_query(query, conn)

# Define the correct phase order based on the evosuite_runtime table
correct_phase_order = [
    'SEARCH',
    'INLINING',
    'MINIMIZATION',
    'COVERAGE_ANALYSIS',
    'ASSERTION_GENERATION',
    'JUNIT_CHECK',
    'WRITING_TESTS',
    'WRITING_STATISTICS',
    'DONE',
    'FINISHED'
]

# Calculate total runtime per project
total_runtime_per_project = df.groupby(['search_budget', 'project_id'])['runtime'].sum().reset_index()
total_runtime_per_project.rename(columns={'runtime': 'total_runtime'}, inplace=True)

# Calculate runtime per phase per project
phase_runtime = df.groupby(['search_budget', 'project_id', 'phase_name'])['runtime'].sum().reset_index()

# Merge the total runtime with phase runtime
phase_analysis = pd.merge(
    phase_runtime, 
    total_runtime_per_project, 
    on=['search_budget', 'project_id']
)

# Calculate the fraction of total runtime for each phase per project
phase_analysis['runtime_fraction'] = phase_analysis['runtime'] / phase_analysis['total_runtime']

# Now average the fractions per search budget and phase
avg_fractions = phase_analysis.groupby(['search_budget', 'phase_name'])['runtime_fraction'].mean().reset_index()

# Create a pivot table with all phases
result_full = avg_fractions.pivot_table(
    index='search_budget',
    columns='phase_name',
    values='runtime_fraction',
    aggfunc='mean'
).fillna(0)

# Get all available phases in the data
available_phases = set(result_full.columns)

# Create a list of phases in the correct order, including only those that exist in the data
ordered_phases = [phase for phase in correct_phase_order if phase in available_phases]

# Add any phases that might be in the data but not in our predefined list (at the end)
extra_phases = [phase for phase in available_phases if phase not in correct_phase_order]
ordered_phases.extend(sorted(extra_phases))

# Reorder the columns based on the correct phase order
result_full = result_full[ordered_phases]

# Format as percentages for better readability
result_full_percentage = result_full.map(lambda x: f"{x*100:.2f}%")

# Display the full results with all phases in the correct order
print("Average fraction of runtime spent in ALL phases per search budget (calculated per project):")
display(result_full_percentage)

# Identify the top 4 phases by average runtime fraction across all search budgets
# Calculate the mean across all rows to find the top phases
phase_means = result_full.mean()
top_phases = phase_means.nlargest(4).index.tolist()

# Create a new DataFrame with only the top 4 phases
# We need to maintain the original order, so filter from ordered_phases
ordered_top_phases = [phase for phase in ordered_phases if phase in top_phases]
result = result_full[ordered_top_phases].copy()

# Add an "OTHER" column that contains the sum of all other phases
other_phases = [phase for phase in result_full.columns if phase not in ordered_top_phases]
result['OTHER'] = result_full[other_phases].sum(axis=1)

# Format as percentages for better readability
result_percentage = result.map(lambda x: f"{x*100:.2f}%")

# Display the results with top 4 phases and OTHER
print("\nAverage fraction of runtime spent in different phases per search budget (calculated per project):")
print("(Showing top 4 phases, others grouped as 'OTHER')")
display(result_percentage)

# Create a stacked bar chart for visualization
plt.figure(figsize=(12, 8))
result.plot(kind='bar', stacked=True, figsize=(12, 8))
plt.title('Average Runtime Fraction per Phase by Search Budget')
plt.xlabel('Search Budget')
plt.ylabel('Fraction of Total Runtime')
plt.legend(title='Phase', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()